# Calculate the spatial gradient of a field on the mesh

**Step 1: Create FELiCSMesh, FELiCSSpace and Field**

This should be familiar from the last How-To. If you need a quick refresher please check [link]

=> not true anymore, explain cylindrical coordinates and wavenumber m
=> also put equation here
=> rename scalarField into "a" or something s.t. the equation looks nice
=> put validation field creation in its own cell and explain shortly (do we need it here?)


In [5]:
from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystemName = "Cylindrical"
m = 3

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystemName=coordinateSystemName, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

scalarSpace = getFELiCSSpace(mesh, order=2, dim=1)
vectorSpace = getFELiCSSpace(mesh, order=2, dim=3)

from FELiCS.Fields.Field import Field

scalarField = Field(scalarSpace, mesh=mesh, m = m)
scalarField.importH5("function_values_sine")
gradientFieldsol = Field(vectorSpace, mesh=mesh)
gradientFieldsol.importH5("grad_solution_cylSpectral_m=3")

Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


**Step 2: Define the gradient Expression and evaluate it on the Field**
We now want to calculate the gradient of our scalar quantity using the `getGradient()` method of our Field class. This method returns a Field 

In [4]:
##################################
# Tensor Utils stuff in Tensor Utils tutorial
from ufl import TestFunction, dx, conj
from FELiCS.Misc.tensorUtils import iGrad, iConj, iDot, Tensor
import dolfinx.fem

J_hat = mesh.coordinateSystem.J_hat
print(scalarSpace.element)
v = TestFunction(vectorSpace)
v_tens = Tensor(v, CoordSys=mesh.coordinateSystem, hasSpectralDimension=True)
expression = iDot(iGrad(scalarField.getTensor()), iConj(v_tens)).ufl_tens * J_hat* dx
###################################

gradientField = scalarField.getGradientField()

print(sum(gradientField.getCoefficientArray() - gradientFieldsol.getCoefficientArray()))

(-0.0521639183515227+2.0653351251453955e-10j)
